### Import Libraries

In [1]:
import pandas as pd
import numpy as np

### Load Cleaned Dataset

In [2]:
data = pd.read_csv("customer_churn_cleaned.csv")

# Check shape and first few rows
print(data.shape)
data.head()

(7043, 22)


,Customer_ID,Gender,Senior_Citizen,Partner,Dependents,Tenure,Phone_Service,Multiple_Lines,Internet_Service,Online_Security,...,Tech_Support,Streaming_TV,Streaming_Movies,Contract,Paperless_Billing,Payment_Method,Monthly_Charges,Total_Charges,Customer_Feedback,Churn_Status
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,Unsatisfied,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,Satisfied,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Unsatisfied,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,Satisfied,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Unsatisfied,Yes


### Encode Categorical Variables

In [3]:
# Copy dataframe to avoid changes
data_ml = data.copy()

# encode target
data_ml['Churn_Encoded'] = data_ml['Churn_Status'].map({'No': 0, 'Yes': 1})

# drop customer ID
X = data_ml.drop(columns = ['Customer_ID'], inplace = True)

# separate features & target
X = data_ml.drop(columns = ['Churn_Status', 'Churn_Encoded'])
y = data_ml['Churn_Encoded']

# Encode categorical columns
X = pd.get_dummies(X, drop_first=True)

### Train-Test Split

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify = y
)

print(X_train.shape, X_test.shape)

(5634, 32) (1409, 32)


### Identify & Scale Numerical Columns

In [5]:
numeric_cols = ['Tenure', 'Monthly_Charges', 'Total_Charges']

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

### Logistic Regression Model

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import roc_auc_score

# Initialize model
log_reg = LogisticRegression(max_iter = 2000)

# Train model
log_reg.fit(X_train_scaled, y_train)

# Make predictions
y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

# Evaluate
print('Logistic Regression Accuracy:', accuracy_score(y_test, y_pred_lr))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred_lr))
print('\nClassification Report:\n', classification_report(y_test, y_pred_lr))
print('\nROC-AUC Score:\n', roc_auc_score(y_test, y_prob_lr))

Logistic Regression Accuracy: 0.8914123491838183

Confusion Matrix:
 [[949  86]
 [ 67 307]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.92      0.93      1035
           1       0.78      0.82      0.80       374

    accuracy                           0.89      1409
   macro avg       0.86      0.87      0.86      1409
weighted avg       0.89      0.89      0.89      1409


ROC-AUC Score:
 0.9607558965615232


### Logistic Regression Coefficients

In [7]:
coef_data = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Coefficient': log_reg.coef_[0]
})

coef_data['Abs_Coefficient'] = coef_data['Coefficient'].abs()
coef_data = coef_data.sort_values(by = 'Abs_Coefficient', ascending = False)

coef_data.head(10)

,Feature,Coefficient,Abs_Coefficient
31,Customer_Feedback_Unsatisfied,6.847046,6.847046
30,Customer_Feedback_Satisfied,-4.004410,4.004410
1,Tenure,1.885240,1.885240
25,Contract_Two year,-1.500760,1.500760
10,Internet_Service_Fiber optic,0.683927,0.683927
24,Contract_One year,-0.589200,0.589200
13,Online_Security_Yes,-0.559825,0.559825
7,Phone_Service_Yes,-0.470689,0.470689
26,Paperless_Billing_Yes,0.425442,0.425442
19,Tech_Support_Yes,-0.401185,0.401185


### Random Forest Model

In [8]:
from sklearn.ensemble import RandomForestClassifier

# Initialize model
rf = RandomForestClassifier(n_estimators = 200, random_state = 42, max_depth = None, class_weight = 'balanced')

# Train model
rf.fit(X_train, y_train)

# Predict
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

# Evaluate
print('Random Forest Accuracy:', accuracy_score(y_test, y_pred_rf))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred_rf))
print('\nClassification Report:\n', classification_report(y_test, y_pred_rf))
print('\nROC-AUC Score:\n', roc_auc_score(y_test, y_prob_rf))

Random Forest Accuracy: 0.8999290276792051

Confusion Matrix:
 [[963  72]
 [ 69 305]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.93      0.93      1035
           1       0.81      0.82      0.81       374

    accuracy                           0.90      1409
   macro avg       0.87      0.87      0.87      1409
weighted avg       0.90      0.90      0.90      1409


ROC-AUC Score:
 0.9655596889612235


### Feature Importance For Random Forest Model

In [9]:
# create dataframe of feature importance
feat_imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf.feature_importances_
})

# sort by importance
feat_imp = feat_imp.sort_values(by = 'Importance', ascending = False)
feat_imp.head(10) # top 10 features

,Feature,Importance
31,Customer_Feedback_Unsatisfied,0.333921
30,Customer_Feedback_Satisfied,0.171964
3,Total_Charges,0.090779
1,Tenure,0.075625
2,Monthly_Charges,0.071238
25,Contract_Two year,0.032722
10,Internet_Service_Fiber optic,0.026415
28,Payment_Method_Electronic check,0.023446
26,Paperless_Billing_Yes,0.012244
4,Gender_Male,0.011134


### Full Model Prediction

In [10]:
# load dataset
data_full = pd.read_csv('customer_churn_cleaned.csv')

# encode target
data_full['Churn_Encoded'] = data_full['Churn_Status'].map({'No': 0, 'Yes': 1})

# separate features & target
X_full = data_full.drop(columns = ['Churn_Status', 'Churn_Encoded', 'Customer_ID'])
y_full = data_full['Churn_Encoded']

# Encode categorical columns
X_full = pd.get_dummies(X_full, drop_first=True)

# scale numeric columns
numeric_cols = ['Tenure', 'Monthly_Charges', 'Total_Charges']

scaler = StandardScaler()
X_full[numeric_cols] = scaler.fit_transform(X_full[numeric_cols])

# train random forest model
final_rf = RandomForestClassifier(n_estimators = 200, random_state = 42, max_depth = None, class_weight = 'balanced')

# Train model
final_rf.fit(X_full, y_full)

# predict churn probabilities
full_probs = final_rf.predict_proba(X_full)[:, 1]

# define risk level
def assign_risk(prob):
    if prob >= 0.7:
        return 'High Risk'
    elif prob >= 0.4:
        return 'Medium Risk'
    else:
        return 'Low Risk'

# assign risk levels to the dataframe
data_full['Risk_Level'] = [assign_risk(p) for p in full_probs]
data_full['Churn_Probability'] = full_probs

print('\nCustomer counts per risk level:')
print(data_full['Risk_Level'].value_counts())


Customer counts per risk level:
Risk_Level
Low Risk       5149
High Risk      1798
Medium Risk      96
Name: count, dtype: int64


### Top 10 High_Risk Customers

In [11]:
top_risk_rf = data_full[['Customer_ID', 'Risk_Level']].copy()
top_risk_rf['Prob_Churn_RF'] = full_probs
top_risk_rf = top_risk_rf.sort_values(by = 'Prob_Churn_RF', ascending = False)

print('\nTop 10 High-Risk Customers:')
print(top_risk_rf.head(10))


Top 10 High-Risk Customers:
     Customer_ID Risk_Level  Prob_Churn_RF
13    0280-XJGEX  High Risk            1.0
1731  8375-DKEBR  High Risk            1.0
7006  0093-XWZFY  High Risk            1.0
8     7892-POOKP  High Risk            1.0
7009  7703-ZEKEF  High Risk            1.0
6914  7142-HVGBG  High Risk            1.0
475   2359-KMGLI  High Risk            1.0
1976  9497-QCMMS  High Risk            1.0
5609  4818-DRBQT  High Risk            1.0
445   7752-XUSCI  High Risk            1.0


### Feature Importance

In [12]:
# create dataframe of feature importance
feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
})

# sort by importance
feat_imp = feat_imp.sort_values(by = 'Importance', ascending = False)
print('\nTop 10 Features by Importance:')
print(feat_imp.head(10)) # top 10 features


Top 10 Features by Importance:
                            Feature  Importance
31    Customer_Feedback_Unsatisfied    0.333921
30      Customer_Feedback_Satisfied    0.171964
3                     Total_Charges    0.090779
1                            Tenure    0.075625
2                   Monthly_Charges    0.071238
25                Contract_Two year    0.032722
10     Internet_Service_Fiber optic    0.026415
28  Payment_Method_Electronic check    0.023446
26            Paperless_Billing_Yes    0.012244
4                       Gender_Male    0.011134


In [13]:
data_full.to_csv('C:/Users/user/Desktop/APROJECT/customer_churn_with_risk_levels.csv', index = False)

In [14]:
data_full.head()

,Customer_ID,Gender,Senior_Citizen,Partner,Dependents,Tenure,Phone_Service,Multiple_Lines,Internet_Service,Online_Security,...,Contract,Paperless_Billing,Payment_Method,Monthly_Charges,Total_Charges,Customer_Feedback,Churn_Status,Churn_Encoded,Risk_Level,Churn_Probability
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,Unsatisfied,No,0,Low Risk,0.205
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,One year,No,Mailed check,56.95,1889.50,Satisfied,No,0,Low Risk,0.000
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Unsatisfied,Yes,1,High Risk,0.775
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,One year,No,Bank transfer (automatic),42.30,1840.75,Satisfied,No,0,Low Risk,0.000
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Unsatisfied,Yes,1,High Risk,0.915
